In [5]:
from operator import add
from typing import TypedDict, Annotated

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState

load_dotenv()

# 1.初始化模型
model = init_chat_model(
    "deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 2.创建State
# class LLMState(TypedDict):
#     messages: Annotated[list[BaseMessage], add]

# 3.创建Node
def llm_node(state: MessagesState):
    # 获取历史消息，也包含最新一次的用户消息
    messages = state.get("messages")
    # 调用模型，传入message消息历史列表，返回类型是AIMessage
    response = model.invoke(messages)
    # 写回State
    return {"messages": [response]}


# 4.添加Edge，连接Node，形成Graph
builder = StateGraph(MessagesState)

builder.add_node("llm", llm_node)

builder.add_edge(START, "llm")
builder.add_edge("llm", END)

graph = builder.compile()

In [6]:
# 5.调用Graph
result = graph.invoke({"messages": [HumanMessage("你好，你是谁？")]})
print(result)

{'messages': [HumanMessage(content='你好，你是谁？', additional_kwargs={}, response_metadata={}, id='660ddb11-acd2-4f7d-a768-555f0faa1fd9'), AIMessage(content='你好！我是DeepSeek，由深度求索公司创造的AI助手。😊\n\n我可以帮你回答各种问题、处理文字信息、进行对话交流。我支持多模态识别，可以接收你上传的图片并分析其中的可见信息，同时支持文件上传功能（图像、txt、pdf、ppt、word、excel等），能从中读取文字信息进行处理。我拥有1M的上下文长度，可以一次性处理像《三体》三部曲那样体量的书籍。\n\n目前我是完全免费的，Web端和App端都可以使用，还支持语音输入功能。如果需要联网搜索，记得手动点开联网搜索按键哦～\n\n有什么我可以帮助你的吗？无论是学习、工作还是生活上的问题，尽管问我！✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 8, 'total_tokens': 160, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '1ba5fda5-95ad-4906-a434-daa8c12fdf52', 'finish_reason': 'stop'